# Agent 1 — Step 4.5: Real-Embedding Calibration for Final-Topic Edit Memory (v2)

## Goal

Step 4 showed that the contextual matcher logic is safe, but the initial fixed gates:

```text
remove / replace = 0.90
change role      = 0.92
add topic        = 0.94
```

are too strict for the actual `sentence-transformers/all-MiniLM-L6-v2` similarity scale.

This notebook **does not lower or change those thresholds automatically**.

Instead, it performs a larger calibration exercise using several positive and negative contexts across different edit types and topics.

The purpose is to answer:

> Is there a conservative similarity region where genuinely compatible human edits score above clearly incompatible teaching contexts?

---

## Important anti-overfitting rule

We do **not** calibrate from only one transcript or one topic.

This notebook uses multiple topic families and multiple wording variants for each edit action.

It also performs a simple **leave-one-anchor-out validation** so that a candidate threshold derived from other examples is tested on an unseen topic/context family.

If the data does not show a safe separation, the notebook reports:

```text
NO_SAFE_THRESHOLD
```

rather than forcing a threshold.

---

## Safety

This notebook:

- does **not** modify Module 1;
- does **not** modify Module 2;
- does **not** modify Module 3;
- does **not** modify the final Module 3 notebook;
- does **not** write to PostgreSQL;
- does **not** modify `topic_mapping_memory`;
- does **not** modify `detected_topic_edit_memory`;
- does **not** change Qdrant;
- does **not** change Groq;
- does **not** change Streamlit;
- does **not** change Agent 2;
- does **not** automatically write new threshold values anywhere.

It is a diagnostic/calibration notebook only.


---

## v2 note — VS Code / Jupyter import-path fix

VS Code's Jupyter kernel can start with a working directory different from the terminal.
This version safely locates the existing `Agent_1` root and adds it to the notebook
kernel's `sys.path` before importing `app.*`.

This changes **only the notebook runtime import path**. It does not modify any Agent 1 code.


## Calibration strategy

For each action family we create several **anchor memories**.

Each anchor has:

- one reviewed evidence context;
- multiple positive paraphrases that should be compatible;
- multiple negative contexts where the same topic words may appear but the human edit should **not** be reused.

The four action families are:

```text
remove_topic
replace_topic
change_role
add_topic
```

### Example

For a removal memory:

```text
Stored context:
"Sorting is only mentioned because binary search requires ordered data."

Positive:
"An unordered list would have to be sorted before binary search."

Negative:
"Bubble sort compares adjacent values and swaps them over repeated passes."
```

The positive and negative examples may share vocabulary, but the teaching context is different.

That is exactly the distinction the self-improving system must learn safely.


In [ ]:
from __future__ import annotations

import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd

from dotenv import load_dotenv


# ---------------------------------------------------------------------
# Project-root bootstrap
# ---------------------------------------------------------------------
# VS Code's Jupyter kernel can use a working directory that is different
# from the terminal's current directory. We therefore locate Agent_1
# explicitly before importing from app.*.
#
# This does NOT modify any project files. It only adds the existing
# Agent_1 project root to this notebook kernel's Python import path.
# ---------------------------------------------------------------------

def find_agent1_root() -> Path:
    cwd = Path.cwd().resolve()

    candidates = []

    # Current directory and its parents.
    candidates.extend([cwd, *cwd.parents])

    # Common case: kernel starts in EDTECH while Agent_1 is a child folder.
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "Agent_1")

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate in seen:
            continue
        seen.add(candidate)

        required_files = [
            candidate / "app",
            candidate / "app" / "services" / "detected_topic_edit_embedding_adapter.py",
            candidate / "app" / "services" / "detected_topic_edit_memory_matcher.py",
        ]

        if all(path.exists() for path in required_files):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Agent_1 project root. "
        "Make sure Step 3 and Step 4 files were copied into "
        "C:\\Users\\hp\\EDTECH\\Agent_1 and run this notebook from "
        "inside the EDTECH project."
    )


AGENT1_ROOT = find_agent1_root()

if str(AGENT1_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT1_ROOT))

# Load the same Agent_1 environment file used by the project.
load_dotenv(AGENT1_ROOT / ".env")


from app.services.detected_topic_edit_embedding_adapter import (
    Agent1EditMemoryEmbeddingAdapter,
)


adapter = Agent1EditMemoryEmbeddingAdapter()

print("Calibration environment loaded.")
print("Agent_1 root  :", AGENT1_ROOT)
print("Kernel cwd    :", Path.cwd().resolve())
print("Embedding model:", adapter.model_name)


# 1. Define calibration cases

These examples are deliberately spread across several Computer Science contexts.

They are **engineering calibration examples**, not production rules.

No topic name or sentence below is hard-coded into the production matcher.


In [ ]:
@dataclass(frozen=True)
class CalibrationAnchor:
    action: str
    anchor_id: str
    stored_evidence: str
    positives: tuple[str, ...]
    negatives: tuple[str, ...]


anchors = [

    # ------------------------------------------------------------------
    # REMOVE TOPIC
    # ------------------------------------------------------------------
    CalibrationAnchor(
        action="remove_topic",
        anchor_id="remove_sorting_incidental",
        stored_evidence=(
            "Binary search requires ordered data. If the list is unsorted, "
            "some form of sorting would be needed before the search can begin."
        ),
        positives=(
            "A binary search only works on ordered items, so an unordered list "
            "would need to be sorted first before binary search is used.",
            "Before using binary search, the data must already be in order; an "
            "unsorted collection would first require sorting.",
            "Sorting is mentioned only because binary search assumes a sorted "
            "list before the search process starts.",
        ),
        negatives=(
            "Bubble sort compares adjacent values and swaps them when they are "
            "in the wrong order until the list becomes sorted.",
            "Merge sort divides the list into smaller sections and merges them "
            "back together in sorted order.",
            "The lesson compares bubble sort and merge sort by tracing their "
            "passes, swaps and sorting behaviour.",
        ),
    ),

    CalibrationAnchor(
        action="remove_topic",
        anchor_id="remove_arithmetic_incidental",
        stored_evidence=(
            "The midpoint is calculated using left plus right and integer "
            "division by two, then the pointers are adjusted by one."
        ),
        positives=(
            "To perform binary search, add the left and right pointer positions, "
            "use integer division by two for the midpoint, and move a pointer.",
            "The search algorithm uses a small arithmetic calculation only to "
            "find the middle index and update the search boundaries.",
            "Addition and division appear only inside the midpoint calculation "
            "used by binary search.",
        ),
        negatives=(
            "This lesson teaches arithmetic operators including addition, "
            "subtraction, multiplication, division, integer division and modulo.",
            "Students practise expressions using multiplication, division and "
            "operator precedence before evaluating arithmetic results.",
            "The topic explains integer division and modulo as programming "
            "operators with several worked examples.",
        ),
    ),

    CalibrationAnchor(
        action="remove_topic",
        anchor_id="remove_data_types_incidental",
        stored_evidence=(
            "An integer variable is used to hold the loop counter while the "
            "lesson focuses on tracing the searching algorithm."
        ),
        positives=(
            "The code uses an integer counter, but the lesson is teaching how "
            "the search loop progresses rather than teaching data types.",
            "A Boolean flag and integer index appear only as implementation "
            "details while the algorithm itself is explained.",
            "Variables have types in the example code, but data types are not "
            "independently explained or practised.",
        ),
        negatives=(
            "The lesson explains integer, real, Boolean and character data types "
            "and when each should be selected.",
            "Students compare data types and choose appropriate types for values "
            "such as age, price, true or false, and single characters.",
            "The topic teaches how values are represented using different "
            "programming data types.",
        ),
    ),

    # ------------------------------------------------------------------
    # REPLACE TOPIC
    # ------------------------------------------------------------------
    CalibrationAnchor(
        action="replace_topic",
        anchor_id="replace_subroutine_statement",
        stored_evidence=(
            "The lesson explains functions and procedures, their parameters, "
            "local variables and returned values, rather than generic statements."
        ),
        positives=(
            "Students learn procedures and functions, including parameters, "
            "return values and local scope.",
            "The teaching is specifically about subroutines: functions return "
            "values while procedures perform tasks.",
            "Parameters and local variables are explained in the context of "
            "procedures and functions.",
        ),
        negatives=(
            "The lesson teaches assignment, selection and iteration statements "
            "as the main programming constructs.",
            "Students identify sequence, IF statements and loops in pseudocode.",
            "The topic focuses on program statements and control structures "
            "without teaching functions or procedures.",
        ),
    ),

    CalibrationAnchor(
        action="replace_topic",
        anchor_id="replace_search_comparison",
        stored_evidence=(
            "The evidence is specifically comparing linear and binary search, "
            "including how each locates an item and how many checks are needed."
        ),
        positives=(
            "The lesson contrasts linear search with binary search and explains "
            "how their search processes differ.",
            "Students compare the steps and efficiency of linear and binary "
            "search when finding the same target.",
            "Both searching algorithms are traced and compared using the same "
            "ordered dataset.",
        ),
        negatives=(
            "The lesson explains generic algorithm properties such as sequence, "
            "selection and iteration without teaching search methods.",
            "Students learn how to write pseudocode for a simple algorithm but "
            "do not compare searching techniques.",
            "The topic introduces algorithms generally as step-by-step solutions "
            "to computational problems.",
        ),
    ),

    # ------------------------------------------------------------------
    # CHANGE ROLE
    # ------------------------------------------------------------------
    CalibrationAnchor(
        action="change_role",
        anchor_id="role_efficiency_primary",
        stored_evidence=(
            "The lesson repeatedly compares two algorithms solving the same "
            "problem and explains why one needs fewer operations and is faster."
        ),
        positives=(
            "Most of the lesson compares algorithms for the same task and "
            "explains their relative execution time and number of operations.",
            "The central teaching objective is comparing algorithm efficiency "
            "and explaining why one solution executes faster than another.",
            "Students repeatedly evaluate which algorithm is more efficient and "
            "justify the comparison using execution behaviour.",
        ),
        negatives=(
            "The teacher briefly notes that binary search is faster than linear "
            "search before returning to the mechanics of the search algorithm.",
            "Efficiency is mentioned once, but the lesson mainly traces pointer "
            "updates in binary search.",
            "The main topic is implementing a search algorithm; speed is only a "
            "short supporting comment.",
        ),
    ),

    CalibrationAnchor(
        action="change_role",
        anchor_id="role_binary_search_primary",
        stored_evidence=(
            "The lesson is centred on binary search: midpoint calculation, "
            "left and right pointers, repeated halving and a full worked trace."
        ),
        positives=(
            "Binary search is taught step by step using midpoint calculations, "
            "pointer movement and repeated halving of the search range.",
            "Most of the lesson traces binary search and explains how the left, "
            "right and middle positions change.",
            "Students work through a full binary-search example and then write "
            "the corresponding search logic.",
        ),
        negatives=(
            "Binary search is briefly named while the lesson mainly compares "
            "general algorithm efficiency.",
            "The main topic is sorting, with binary search mentioned only as a "
            "possible use of ordered data.",
            "Searching is a supporting example while the lesson focuses on "
            "Boolean logic and loop conditions.",
        ),
    ),

    CalibrationAnchor(
        action="change_role",
        anchor_id="role_arrays_primary",
        stored_evidence=(
            "The lesson primarily teaches one-dimensional and two-dimensional "
            "arrays, including indexing, traversal and storing multiple values."
        ),
        positives=(
            "Students spend the lesson creating and traversing arrays and "
            "accessing elements using indexes.",
            "The main teaching focus is array structures, including rows, "
            "columns, indexes and iteration over stored values.",
            "Most examples use arrays to explain how multiple values are stored "
            "and retrieved.",
        ),
        negatives=(
            "An array is used only as the dataset for demonstrating binary "
            "search, while the lesson focuses on the search algorithm.",
            "The code stores values in an array, but the teaching objective is "
            "sorting the values with bubble sort.",
            "Arrays appear in an example program while the lesson mainly teaches "
            "subroutines and parameters.",
        ),
    ),

    # ------------------------------------------------------------------
    # ADD TOPIC
    # ------------------------------------------------------------------
    CalibrationAnchor(
        action="add_topic",
        anchor_id="add_boolean_explicit",
        stored_evidence=(
            "The lesson explicitly teaches Boolean AND OR and NOT operators, "
            "including truth conditions and worked programming examples."
        ),
        positives=(
            "Students are taught how AND OR and NOT work with truth values and "
            "apply the Boolean operators in several code examples.",
            "The lesson explains Boolean operators with truth conditions and "
            "practises combining Boolean expressions.",
            "AND, OR and NOT are taught directly using Boolean values and "
            "worked logical expressions.",
        ),
        negatives=(
            "A found flag is set to false and checked in a while loop while the "
            "main lesson explains binary search pointer updates.",
            "A Boolean variable appears in the program but Boolean logic is not "
            "explained independently.",
            "The code checks a true or false condition only as part of a loop "
            "used to demonstrate another algorithm.",
        ),
    ),

    CalibrationAnchor(
        action="add_topic",
        anchor_id="add_parameters_explicit",
        stored_evidence=(
            "The lesson explicitly explains parameters, arguments and return "
            "values when calling functions and procedures."
        ),
        positives=(
            "Students learn how parameters pass values into subroutines and how "
            "functions return results.",
            "The teaching directly covers function parameters, arguments and "
            "returned values with worked calls.",
            "Several examples explain parameter lists and the values returned "
            "from functions.",
        ),
        negatives=(
            "A function call contains two parameters, but the lesson focuses on "
            "sorting the array passed into the function.",
            "Parameters appear in example code while the teacher mainly explains "
            "searching algorithms.",
            "A procedure receives a value, but parameter passing is not discussed "
            "as an independent topic.",
        ),
    ),

    CalibrationAnchor(
        action="add_topic",
        anchor_id="add_validation_explicit",
        stored_evidence=(
            "The lesson teaches validation routines such as range checks, type "
            "checks and presence checks, with examples of invalid input."
        ),
        positives=(
            "Students learn range, type and presence checks and apply them to "
            "validate user input.",
            "The topic explicitly explains several validation methods and shows "
            "how bad input is rejected.",
            "Input validation is taught through range checks and other routines "
            "with worked examples.",
        ),
        negatives=(
            "The program checks whether a search index is within range, but the "
            "lesson is about binary search rather than input validation.",
            "An IF statement rejects one invalid value while the main topic is "
            "selection and iteration.",
            "A boundary check appears inside an algorithm without teaching "
            "validation routines as a separate concept.",
        ),
    ),
]

print("Calibration anchors:", len(anchors))
print("Action counts:")
pd.Series([a.action for a in anchors]).value_counts()


# 2. Embed all calibration texts using the real Agent 1 embedding model

All text is embedded in one batch to keep the diagnostic reproducible and efficient.


In [ ]:
def cosine_similarity(left: Sequence[float], right: Sequence[float]) -> float:
    left = np.asarray(left, dtype=np.float32)
    right = np.asarray(right, dtype=np.float32)

    left_norm = np.linalg.norm(left)
    right_norm = np.linalg.norm(right)

    if left_norm == 0 or right_norm == 0:
        return 0.0

    return float(np.dot(left, right) / (left_norm * right_norm))


all_texts = []
text_index = {}

def register(text: str) -> None:
    if text not in text_index:
        text_index[text] = len(all_texts)
        all_texts.append(text)

for anchor in anchors:
    register(anchor.stored_evidence)
    for text in anchor.positives:
        register(text)
    for text in anchor.negatives:
        register(text)

print("Unique texts to embed:", len(all_texts))

vectors = adapter.embed_texts(all_texts)

print("Vectors returned:", len(vectors))
print("Embedding dimension:", len(vectors[0]))


# 3. Build the similarity dataset

Each positive and negative example is compared only with its own reviewed anchor evidence.

The output table records:

```text
action
anchor_id
label = positive / negative
similarity
```


In [ ]:
rows = []

for anchor in anchors:
    anchor_vector = vectors[text_index[anchor.stored_evidence]]

    for text in anchor.positives:
        similarity = cosine_similarity(
            anchor_vector,
            vectors[text_index[text]],
        )
        rows.append(
            {
                "action": anchor.action,
                "anchor_id": anchor.anchor_id,
                "label": "positive",
                "similarity": similarity,
                "text": text,
            }
        )

    for text in anchor.negatives:
        similarity = cosine_similarity(
            anchor_vector,
            vectors[text_index[text]],
        )
        rows.append(
            {
                "action": anchor.action,
                "anchor_id": anchor.anchor_id,
                "label": "negative",
                "similarity": similarity,
                "text": text,
            }
        )

df = pd.DataFrame(rows)

print("Calibration comparisons:", len(df))
df.head(10)


# 4. Inspect positive and negative distributions

For each action we calculate:

- minimum positive similarity;
- 10th percentile positive similarity;
- median positive similarity;
- maximum negative similarity;
- 90th percentile negative similarity;
- separation gap.

A positive gap is useful, but one small calibration set is still not enough to justify automatic deployment by itself.


In [ ]:
summary_rows = []

for action, group in df.groupby("action"):
    positives = group.loc[
        group["label"] == "positive",
        "similarity",
    ].to_numpy()

    negatives = group.loc[
        group["label"] == "negative",
        "similarity",
    ].to_numpy()

    positive_min = float(np.min(positives))
    positive_p10 = float(np.quantile(positives, 0.10))
    positive_median = float(np.median(positives))

    negative_max = float(np.max(negatives))
    negative_p90 = float(np.quantile(negatives, 0.90))
    negative_median = float(np.median(negatives))

    summary_rows.append(
        {
            "action": action,
            "positive_min": positive_min,
            "positive_p10": positive_p10,
            "positive_median": positive_median,
            "negative_median": negative_median,
            "negative_p90": negative_p90,
            "negative_max": negative_max,
            "strict_gap_pos_min_minus_neg_max": (
                positive_min - negative_max
            ),
            "robust_gap_p10_minus_p90": (
                positive_p10 - negative_p90
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("action")

summary_df.round(4)


# 5. Candidate-threshold analysis — diagnostic only

We prioritize **precision**.

A threshold is considered a possible candidate only when there is empirical separation between positive and negative calibration examples.

The notebook does not write the result anywhere.

### Candidate rule

For each action:

1. Calculate the highest negative similarity.
2. Add a small safety margin.
3. Confirm that this value still sits below the lower positive region.
4. If not, report `NO_SAFE_THRESHOLD`.

This is intentionally conservative.


In [ ]:
SAFETY_MARGIN = 0.03

candidate_rows = []

for _, row in summary_df.iterrows():
    action = row["action"]

    negative_ceiling = float(row["negative_max"])
    positive_floor = float(row["positive_p10"])

    candidate = negative_ceiling + SAFETY_MARGIN

    if candidate < positive_floor:
        status = "CANDIDATE_AVAILABLE"
        threshold = candidate
    else:
        status = "NO_SAFE_THRESHOLD"
        threshold = np.nan

    candidate_rows.append(
        {
            "action": action,
            "negative_max": negative_ceiling,
            "positive_p10": positive_floor,
            "safety_margin": SAFETY_MARGIN,
            "candidate_threshold": threshold,
            "status": status,
        }
    )

candidate_df = pd.DataFrame(candidate_rows)

candidate_df.round(4)


# 6. Evaluate candidate thresholds on the full calibration set

This gives descriptive:

- positive recall;
- negative rejection rate;
- false-positive count;
- false-negative count.

For this feature, **false positives are more dangerous than false negatives**.

A false negative simply causes the reviewer to edit the topic again.

A false positive could automatically remove, replace, add, or re-role a topic incorrectly.


In [ ]:
evaluation_rows = []

for _, candidate in candidate_df.iterrows():
    action = candidate["action"]
    threshold = candidate["candidate_threshold"]

    action_df = df[df["action"] == action].copy()

    if pd.isna(threshold):
        evaluation_rows.append(
            {
                "action": action,
                "threshold": np.nan,
                "positive_recall": np.nan,
                "negative_rejection_rate": np.nan,
                "false_positives": np.nan,
                "false_negatives": np.nan,
                "status": "NO_SAFE_THRESHOLD",
            }
        )
        continue

    action_df["predicted_hit"] = (
        action_df["similarity"] >= threshold
    )

    positives = action_df[action_df["label"] == "positive"]
    negatives = action_df[action_df["label"] == "negative"]

    false_positives = int(negatives["predicted_hit"].sum())
    false_negatives = int((~positives["predicted_hit"]).sum())

    evaluation_rows.append(
        {
            "action": action,
            "threshold": float(threshold),
            "positive_recall": float(
                positives["predicted_hit"].mean()
            ),
            "negative_rejection_rate": float(
                (~negatives["predicted_hit"]).mean()
            ),
            "false_positives": false_positives,
            "false_negatives": false_negatives,
            "status": (
                "PRECISION_SAFE_ON_CALIBRATION"
                if false_positives == 0
                else "REVIEW_REQUIRED"
            ),
        }
    )

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df.round(4)


# 7. Leave-one-anchor-out validation

This is the most important anti-overfitting check in this notebook.

For each anchor/context family:

1. remove that anchor from calibration;
2. derive a threshold using the remaining anchors of the same action;
3. test the held-out positive and negative examples.

This asks whether a threshold learned from other topic/context families generalizes to an unseen family.

If an action has too few independent anchors, the notebook reports that more calibration data is required.


In [ ]:
loo_rows = []

for action in sorted(df["action"].unique()):
    action_df = df[df["action"] == action]
    anchor_ids = sorted(action_df["anchor_id"].unique())

    for held_out_anchor in anchor_ids:
        training = action_df[
            action_df["anchor_id"] != held_out_anchor
        ]
        testing = action_df[
            action_df["anchor_id"] == held_out_anchor
        ]

        training_anchors = training["anchor_id"].nunique()

        if training_anchors < 1:
            loo_rows.append(
                {
                    "action": action,
                    "held_out_anchor": held_out_anchor,
                    "threshold": np.nan,
                    "false_positives": np.nan,
                    "false_negatives": np.nan,
                    "status": "INSUFFICIENT_TRAINING_ANCHORS",
                }
            )
            continue

        train_pos = training.loc[
            training["label"] == "positive",
            "similarity",
        ].to_numpy()

        train_neg = training.loc[
            training["label"] == "negative",
            "similarity",
        ].to_numpy()

        positive_floor = float(
            np.quantile(train_pos, 0.10)
        )
        negative_ceiling = float(
            np.max(train_neg)
        )

        threshold = negative_ceiling + SAFETY_MARGIN

        if threshold >= positive_floor:
            loo_rows.append(
                {
                    "action": action,
                    "held_out_anchor": held_out_anchor,
                    "threshold": threshold,
                    "false_positives": np.nan,
                    "false_negatives": np.nan,
                    "status": "NO_SAFE_TRAINING_THRESHOLD",
                }
            )
            continue

        predicted = (
            testing["similarity"] >= threshold
        )

        false_positives = int(
            (
                (testing["label"] == "negative")
                & predicted
            ).sum()
        )

        false_negatives = int(
            (
                (testing["label"] == "positive")
                & (~predicted)
            ).sum()
        )

        loo_rows.append(
            {
                "action": action,
                "held_out_anchor": held_out_anchor,
                "threshold": threshold,
                "false_positives": false_positives,
                "false_negatives": false_negatives,
                "status": (
                    "PASS"
                    if false_positives == 0
                    else "FAIL_FALSE_POSITIVE"
                ),
            }
        )

loo_df = pd.DataFrame(loo_rows)

loo_df.round(4)


# 8. Leave-one-anchor-out summary

For automatic final-topic edits, the key metric is:

```text
held-out false positives = 0
```

False negatives are acceptable at this stage because they simply mean the system abstains and asks for human review again.


In [ ]:
loo_summary_rows = []

for action, group in loo_df.groupby("action"):
    usable = group[
        group["status"].isin(
            ["PASS", "FAIL_FALSE_POSITIVE"]
        )
    ]

    if usable.empty:
        loo_summary_rows.append(
            {
                "action": action,
                "held_out_tests": 0,
                "total_false_positives": np.nan,
                "total_false_negatives": np.nan,
                "precision_safety": "NOT_ENOUGH_EVIDENCE",
            }
        )
        continue

    total_fp = int(
        usable["false_positives"].fillna(0).sum()
    )
    total_fn = int(
        usable["false_negatives"].fillna(0).sum()
    )

    loo_summary_rows.append(
        {
            "action": action,
            "held_out_tests": len(usable),
            "total_false_positives": total_fp,
            "total_false_negatives": total_fn,
            "precision_safety": (
                "PASS"
                if total_fp == 0
                else "FAIL"
            ),
        }
    )

loo_summary_df = pd.DataFrame(loo_summary_rows)

loo_summary_df


# 9. Detailed highest-risk negative examples

These are the negative examples most likely to become false automatic edits.

Reviewing them manually is useful before any production threshold is adopted.


In [ ]:
highest_risk_negatives = (
    df[df["label"] == "negative"]
    .sort_values(
        ["action", "similarity"],
        ascending=[True, False],
    )
    .groupby("action", as_index=False)
    .head(5)
    [
        [
            "action",
            "anchor_id",
            "similarity",
            "text",
        ]
    ]
)

highest_risk_negatives.round(4)


# 10. Detailed lowest-scoring positive paraphrases

These examples show where the embedding model struggles to recognize a valid compatible context.

A low positive score is not automatically a reason to lower the threshold.

It may be safer to accept a memory miss than to increase false-positive reuse.


In [ ]:
lowest_positive_examples = (
    df[df["label"] == "positive"]
    .sort_values(
        ["action", "similarity"],
        ascending=[True, True],
    )
    .groupby("action", as_index=False)
    .head(5)
    [
        [
            "action",
            "anchor_id",
            "similarity",
            "text",
        ]
    ]
)

lowest_positive_examples.round(4)


# 11. Final calibration readiness decision

The notebook uses conservative rules.

An action is **not** considered ready merely because a candidate threshold exists.

For this calibration exercise we require:

1. candidate threshold exists;
2. full calibration has zero false positives;
3. leave-one-anchor-out validation has zero false positives;
4. at least two independent anchor families exist for that action.

If any requirement fails, the result remains:

```text
NOT_READY
```

No code or threshold is changed automatically.


In [ ]:
readiness_rows = []

for action in sorted(df["action"].unique()):
    candidate_row = candidate_df[
        candidate_df["action"] == action
    ].iloc[0]

    evaluation_row = evaluation_df[
        evaluation_df["action"] == action
    ].iloc[0]

    loo_row = loo_summary_df[
        loo_summary_df["action"] == action
    ].iloc[0]

    anchor_count = int(
        df[df["action"] == action]["anchor_id"].nunique()
    )

    candidate_exists = (
        candidate_row["status"] == "CANDIDATE_AVAILABLE"
    )

    full_zero_fp = (
        evaluation_row["status"]
        == "PRECISION_SAFE_ON_CALIBRATION"
    )

    loo_zero_fp = (
        loo_row["precision_safety"] == "PASS"
    )

    enough_anchors = anchor_count >= 2

    ready = all(
        [
            candidate_exists,
            full_zero_fp,
            loo_zero_fp,
            enough_anchors,
        ]
    )

    readiness_rows.append(
        {
            "action": action,
            "independent_anchors": anchor_count,
            "candidate_threshold": (
                candidate_row["candidate_threshold"]
                if candidate_exists
                else np.nan
            ),
            "candidate_exists": candidate_exists,
            "full_calibration_zero_false_positives": full_zero_fp,
            "leave_one_anchor_out_zero_false_positives": loo_zero_fp,
            "enough_independent_anchors": enough_anchors,
            "calibration_status": (
                "READY_FOR_REVIEW"
                if ready
                else "NOT_READY"
            ),
        }
    )

readiness_df = pd.DataFrame(readiness_rows)

readiness_df.round(4)


# 12. Human review checkpoint

## Do not wire anything yet

After running the notebook, inspect these outputs:

1. `summary_df`
2. `candidate_df`
3. `evaluation_df`
4. `loo_df`
5. `loo_summary_df`
6. `highest_risk_negatives`
7. `lowest_positive_examples`
8. `readiness_df`

Send the tables/screenshots for review before changing `EditMemoryMatchConfig`.

### Why?

Even if a threshold looks numerically good on these examples, we still want to verify that:

- the highest-risk negatives make semantic sense;
- the lowest positives are genuinely valid reuse cases;
- one topic family is not dominating the threshold;
- false-positive risk remains effectively zero.

---

## What happens after calibration

Only if the evidence is convincing will we create a **versioned edit-memory threshold configuration** and regression tests.

Then, and only then, we move to the actual overlay wiring.

The desired final architecture remains:

```text
Fresh Module 3 output
        ↓
reviewer-approved edit-memory lookup
        ↓
same spec + same source concept where applicable
        ↓
strong contextual compatibility
        ↓
unambiguous?
   ┌────┴────┐
   │         │
  YES        NO
   │         │
apply edit   leave Module 3 untouched
   │
final topic list
   ↓
Agent 2 handoff
```

A memory miss is safe.

An incorrect automatic human edit is not.

That is why this mechanism is intentionally precision-first.
